# KMeans — trajetória por Regional de Saúde

Unidade de agrupamento: **Regional de Saúde** (28 linhas). O tempo não é colapsado em média/tendência -- cada combinação (categoria, ano) vira uma coluna própria, preservando a série completa como estrutura larga (mesma ideia do notebook painel, mas sem colapsar em pares Regional-Ano).

Mesmas regras de decisão do DBSCAN/KMeans painel (`utilitarios_agrupamento_kmeansManual.py`, esta pasta) -- montagem a partir dos CSVs de taxa, nenhum nulo imputado, RobustScaler + PCA condicional -- com dois ajustes específicos da trajetória:

- Com 28 regionais e potencialmente muitas colunas (categoria x ano), o regime é `p >> n` -- por isso `USAR_JANELA_COMUM=True` é o padrão aqui (reduz o número de anos, logo de colunas), e o PCA deixa de ser opcional na prática.
- Cada linha é uma REGIONAL inteira -- descartar uma linha custa muito mais caro que no painel. Por isso o corte de ausentes é feito por COLUNA primeiro (`relatorio_de_ausentes` + `MAXIMO_DE_AUSENTES_PARA_DESCARTAR`), e o descarte de linha é a última opção, com o log dizendo qual regional saiu.

Toda a lógica reutilizável está em `utilitarios_agrupamento_kmeansManual.py` e `graficos_agrupamento_kmeansManual.py` (esta pasta, independentes de `DBSCAN/` e compartilhados só com o notebook painel de KMeans); este notebook só configura e executa. Nenhum resultado é gravado em disco.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))


from utilitarios_agrupamento_kmeansManual import (
    COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    VARIANCIA_EXPLICADA_ALVO, NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA,
    ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    SEMENTES_DA_BUSCA,
    montar_painel, calcular_janela_comum, restringir_a_janela_comum,
    descartar_colunas_sem_informacao_e_total,
    converter_blocos_em_proporcao, colapsar_painel_em_trajetoria,
    relatorio_de_ausentes, descartar_colunas_constantes,
    padronizar_e_reduzir, preparar_projecao_2d, pesos_das_variaveis_no_componente,
    calcular_inercia_e_silhueta_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    diagnostico_por_regional,
    montar_resumo_parametros_kmeans,
    registrar_log,
)
from graficos_agrupamento_kmeansManual import (
    grafico_variancia_explicada, grafico_pesos_da_componente,
    grafico_cotovelo_inercia, grafico_silhueta_por_k,
    grafico_dispersao_regionais_kmeans, grafico_perfil_dos_clusters,
)

## 1. Configuração

`USAR_JANELA_COMUM=True` (padrão desta análise, diferente do painel) restringe todos os blocos à interseção `ANO_INICIAL_COMUM`-`ANO_FINAL_COMUM` antes de colapsar em trajetória -- sem isso, o número de colunas (categoria x ano) explode.

`CONVERTER_EM_PROPORCAO=True` converte cada bloco em proporção dentro do bloco (por regional-ano), para o porte da regional não dominar a distância -- ligue/desligue para comparar; com `False`, espere PC1 concentrando a maior parte da variância e Belo Horizonte isolada num extremo (mesmo alerta automático do painel).

In [2]:
USAR_JANELA_COMUM = True
CONVERTER_EM_PROPORCAO = True

# Cada linha é uma REGIONAL inteira -- descartar uma linha custa muito mais
# que no painel. Por isso o corte é por COLUNA primeiro, e o descarte de
# linha é a última opção (com o log dizendo qual regional saiu).
MAXIMO_DE_AUSENTES_PARA_DESCARTAR = 0.20

# Com poucas regionais sobreviventes ao descarte de ausentes, um K_MAXIMO
# alto não faz sentido -- k é também, na prática, o tamanho mínimo médio
# de cluster. Ajustado abaixo conforme o número de linhas que sobrar.
K_MINIMO = 2
K_MAXIMO = 8

## 2. Leitura dos CSVs de taxa e colapso em trajetória (uma coluna por categoria x ano)

In [3]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)

if USAR_JANELA_COMUM:
    janela = calcular_janela_comum(BLOCOS_DE_TAXA, ano_inicial_manual=ANO_INICIAL_COMUM, ano_final_manual=ANO_FINAL_COMUM)
    painel = restringir_a_janela_comum(painel, janela)
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")

painel = descartar_colunas_sem_informacao_e_total(painel)

# Proporção é aplicada ANTES do colapso em trajetória, para ser calculada
# dentro de cada (bloco, regional, ano) -- e não misturando anos diferentes.
if CONVERTER_EM_PROPORCAO:
    painel = converter_blocos_em_proporcao(painel, prefixos=[bloco["PREFIXO"] for bloco in BLOCOS_DE_TAXA])

matriz = colapsar_painel_em_trajetoria(painel)
matriz.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).
[LOG] Janela comum: 2013-2023 (definida manualmente).
[LOG] Restrita à janela comum 2013-2023: 308 de 1316 par(es) regional-ano.
[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Bloco 'casos-faixa': 8 coluna(s) convertida(s) em proporção.
[LOG] Bloco 'casos-escolaridade': 

[LOG] Bloco 'mortalidade': 8 coluna(s) convertida(s) em proporção.
[LOG] Bloco 'mamografia': 8 coluna(s) convertida(s) em proporção.
[LOG] Observação: as proporções de um bloco somam 1, então a última categoria de cada bloco é combinação linear das demais. O PCA absorve essa redundância, mas se você interpretar as variáveis diretamente, considere descartar uma categoria de referência.
[LOG] Trajetória: 28 regional(is) x 385 variável(is) (categoria x ano).
[LOG] ATENÇÃO: 385 variáveis para 28 observações. O número máximo de componentes principais com informação é 27 -- relate essa limitação na monografia.


,casos-faixa_0-9_2013,casos-faixa_0-9_2014,casos-faixa_0-9_2015,casos-faixa_0-9_2016,casos-faixa_0-9_2017,casos-faixa_0-9_2018,casos-faixa_0-9_2019,casos-faixa_0-9_2020,casos-faixa_0-9_2021,casos-faixa_0-9_2022,...,mamografia_70-ou-mais_2014,mamografia_70-ou-mais_2015,mamografia_70-ou-mais_2016,mamografia_70-ou-mais_2017,mamografia_70-ou-mais_2018,mamografia_70-ou-mais_2019,mamografia_70-ou-mais_2020,mamografia_70-ou-mais_2021,mamografia_70-ou-mais_2022,mamografia_70-ou-mais_2023
REGIONAL DE SAÚDE,,,,,,,,,,,,,,,,,,,,,
alfenas,0.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,0.000000,0.0,0.0,...,0.010774,0.070833,0.080760,0.068984,0.063195,0.070909,0.065046,0.064103,0.064180,0.072893
barbacena,0.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,0.000000,0.0,0.0,...,0.031274,0.061815,0.069190,0.054023,0.066172,0.076292,0.076786,0.067206,0.086256,0.084304
belo horizonte,0.000345,NaN,0.000419,NaN,NaN,0.001843,NaN,0.000553,0.0,0.0,...,0.125667,0.120991,0.089832,0.101169,0.096818,0.095597,0.089540,0.084718,0.084205,0.085199
coronel fabriciano,0.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,0.000000,0.0,0.0,...,0.054767,0.082046,0.056755,0.039155,0.051700,0.053163,0.055790,0.060874,0.067316,0.068467
diamantina,0.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,0.000000,0.0,0.0,...,0.043568,0.087400,0.023993,0.024055,0.010473,0.017883,0.015266,0.014052,0.021479,0.019867


## 3. Descarte de colunas e regionais com muito ausente

Nos CSVs de taxa, NaN nunca vira 0 (mesma regra do painel) -- não dá para distinguir "zero eventos" de "sem população válida para calcular" só olhando a taxa. Aqui o corte é em duas etapas: primeiro colunas (categoria x ano) com ausência acima de `MAXIMO_DE_AUSENTES_PARA_DESCARTAR`, depois as regionais que ainda restarem com algum valor ausente -- descartar uma regional inteira é caro, então a ordem importa.

In [4]:
relatorio = relatorio_de_ausentes(matriz)
relatorio.head(20)

,prop_ausente,prop_zero,n_zero
mortalidade_0-9_2022,1.0,0.0,0
casos-faixa_0-9_2019,1.0,0.0,0
mortalidade_0-9_2021,1.0,0.0,0
mortalidade_0-9_2020,1.0,0.0,0
mortalidade_0-9_2019,1.0,0.0,0
mortalidade_0-9_2018,1.0,0.0,0
casos-faixa_10-19_2013,1.0,0.0,0
mortalidade_0-9_2017,1.0,0.0,0
casos-faixa_0-9_2016,1.0,0.0,0
mortalidade_0-9_2016,1.0,0.0,0


In [5]:
colunas_muito_ausentes = relatorio.index[relatorio["prop_ausente"] >= MAXIMO_DE_AUSENTES_PARA_DESCARTAR].tolist()
if colunas_muito_ausentes:
    registrar_log(
        f"{len(colunas_muito_ausentes)} coluna(s) com ausência >= "
        f"{MAXIMO_DE_AUSENTES_PARA_DESCARTAR:.0%} serão descartada(s)."
    )
    matriz = matriz.drop(columns=colunas_muito_ausentes)

if matriz.shape[1] == 0:
    raise ValueError(
        "Nenhuma coluna sobreviveu ao critério de ausentes. Relaxe "
        "MAXIMO_DE_AUSENTES_PARA_DESCARTAR, ligue USAR_JANELA_COMUM, ou reduza a janela comum."
    )

regionais_com_ausente = matriz.index[matriz.isna().any(axis=1)].tolist()
if regionais_com_ausente:
    registrar_log(
        f"{len(regionais_com_ausente)} regional(is) ainda com valor ausente -- "
        f"serão descartada(s) do agrupamento: {regionais_com_ausente}"
    )
matriz = matriz.dropna()

if matriz.shape[0] == 0:
    raise ValueError(
        "Nenhuma regional sobreviveu ao descarte de ausentes -- as colunas que passaram no "
        "critério não têm, juntas, nenhuma linha completa."
    )

matriz = descartar_colunas_constantes(matriz)
registrar_log(f"Matriz final: {matriz.shape[0]} regional(is) x {matriz.shape[1]} variável(is).")
if matriz.shape[0] < 10:
    registrar_log(
        "ATENÇÃO: sobraram pouquíssimas regionais. Com essa amostra, qualquer agrupamento "
        "encontrado é frágil -- relate o tamanho da amostra junto com o resultado."
    )
K_MAXIMO = min(K_MAXIMO, matriz.shape[0] - 1)

[LOG] 50 coluna(s) com ausência >= 20% serão descartada(s).
[LOG] 14 regional(is) ainda com valor ausente -- serão descartada(s) do agrupamento: ['barbacena', 'diamantina', 'itabira', 'ituiutaba', 'januaria', 'manhumirim', 'passos', 'pedra azul', 'pirapora', 'ponte nova', 'sao joao del rei', 'sete lagoas', 'teofilo otoni', 'unai']
[LOG] Descartada(s) 2 coluna(s) constante(s): ['casos-faixa_0-9_2022', 'casos-raca-cor_indigena_2015']
[LOG] Matriz final: 14 regional(is) x 333 variável(is).


## 4. Padronização (RobustScaler) e PCA

In [6]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=True,  # regime p >> n aqui -- PCA sempre necessário, diferente do painel
)

[LOG] PCA aplicado: USAR_PCA=True (forçado; razão atributos/linhas = 23.786) -> 9 componente(s), 88.0% da variância acumulada.


In [7]:
if pca is not None:
    grafico_variancia_explicada(pca, variancia_acumulada).show()
    display(pesos_das_variaveis_no_componente(pca, matriz.columns, componente=1))
    grafico_pesos_da_componente(pca, matriz.columns, componente=1).show()
else:
    registrar_log("USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica.")

casos-escolaridade_nenhuma_2018                 4.276101e-01
casos-escolaridade_fundamental-completo_2014    2.458266e-01
casos-escolaridade_fundamental-completo_2015    2.391462e-01
casos-escolaridade_nenhuma_2014                 2.133967e-01
casos-escolaridade_nenhuma_2017                 1.754657e-01
                                                    ...     
casos-raca-cor_indigena_2013                   -1.904255e-06
casos-faixa_0-9_2020                           -1.600244e-06
casos-faixa_0-9_2013                           -9.995178e-07
casos-faixa_10-19_2014                         -9.166567e-07
casos-faixa_10-19_2023                         -1.400129e-07
Name: PC1, Length: 333, dtype: float64

## 5. Escolha de k (cotovelo da curva de inércia)

In [8]:
tabela_k = calcular_inercia_e_silhueta_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k

[LOG] Inércia e silhueta calculadas para k = 2..8 (7 valor(es) de k).
[LOG] ATENÇÃO: kneed não encontrou cotovelo na curva de inércia -- usando o k de maior curvatura discreta (segunda derivada) como reserva.
[LOG] Cotovelo da curva de inércia na posição 4 de 7 -- k = 6


,k,inercia,silhueta
0,2,2932.233492,0.288596
1,3,2405.952883,0.218972
2,4,1914.270667,0.203738
3,5,1519.054532,0.183363
4,6,1145.363987,0.133659
5,7,865.513926,0.136963
6,8,631.290468,0.145838


### Silhueta por k (validação complementar)

In [9]:
figura_silhueta = grafico_silhueta_por_k(tabela_k, k_escolhido)
figura_silhueta.show()

### Busca em grade (validação complementar)

In [10]:
tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
tabela_da_busca.head(20)

,k,semente,inercia,silhueta
0,2,42,2932.233492,0.288596
1,3,42,2405.952883,0.218972
2,4,42,1914.270667,0.203738
3,5,42,1519.054532,0.183363
4,6,42,1145.363987,0.133659
5,7,42,865.513926,0.136963
6,8,42,631.290468,0.145838


## 6. KMeans

In [11]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA)
n_clusters, inercia_final, silhueta_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")
registrar_log(f"  Silhueta: {silhueta_final:.4f}" if not np.isnan(silhueta_final) else "  Silhueta: não aplicável (k < 2)")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=3)

resultado = pd.DataFrame({COLUNA_REGIONAL: matriz.index, "CLUSTER": rotulos})
resultado.sort_values(["CLUSTER", COLUNA_REGIONAL])

[LOG] KMeans(k=6, random_state=42)
[LOG]   Agrupamentos: 6
[LOG]   Inércia (WCSS): 1145.3640
[LOG]   Silhueta: 0.1337
[LOG] ATENÇÃO: 5 cluster(s) com menos de 3 elemento(s) -- {0: 1, 2: 1, 3: 1, 4: 2, 5: 1}. O KMeans aloca todo ponto a algum cluster (não existe 'ruído' como no DBSCAN); um cluster muito pequeno tende a ser um outlier isolado, não um perfil compartilhado -- considere reduzir k ou tratar esses pontos como atípicos na interpretação.


,REGIONAL DE SAÚDE,CLUSTER
0,alfenas,0
1,belo horizonte,1
3,divinopolis,1
5,juiz de fora,1
9,pouso alegre,1
10,uba,1
11,uberaba,1
12,uberlandia,1
13,varginha,1
2,coronel fabriciano,2


### Composição de cada cluster

In [12]:
resumo_por_cluster = diagnostico_por_regional(resultado)
resumo_por_cluster

[LOG] Cluster 0 (1): alfenas
[LOG] Cluster 1 (8): belo horizonte, divinopolis, juiz de fora, pouso alegre, uba, uberaba, uberlandia, varginha
[LOG] Cluster 2 (1): coronel fabriciano
[LOG] Cluster 3 (1): leopoldina
[LOG] Cluster 4 (2): governador valadares, montes claros
[LOG] Cluster 5 (1): patos de minas


,CLUSTER,regionais,nomes
0,0,1,alfenas
1,1,8,"belo horizonte, divinopolis, juiz de fora, pou..."
2,2,1,coronel fabriciano
3,3,1,leopoldina
4,4,2,"governador valadares, montes claros"
5,5,1,patos de minas


## Visualizações

In [13]:
matriz_2d, variancia_explicada_2d = preparar_projecao_2d(matriz_padronizada, matriz_para_kmeans, pca, semente=SEMENTE_ALEATORIA)

figura_dispersao = grafico_dispersao_regionais_kmeans(matriz_2d, rotulos, matriz.index, variancia_explicada_2d, k_escolhido)
figura_dispersao.show()

In [14]:
figura_perfil = grafico_perfil_dos_clusters(matriz_padronizada, matriz.columns, matriz.index, rotulos)
figura_perfil.show()

## Saídas finais

In [15]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final, silhueta_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros

,K,ESCALONADOR,PCA_APLICADO,PCA_N_COMPONENTES,PCA_VARIANCIA_EXPLICADA,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA,SILHUETA
0,6,RobustScaler,True,9,0.87988,14,333,6,1145.363987,0.133659


In [16]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_0-9_2013
1,casos-faixa_0-9_2015
2,casos-faixa_0-9_2018
3,casos-faixa_0-9_2020
4,casos-faixa_0-9_2021
...,...
328,mamografia_70-ou-mais_2019
329,mamografia_70-ou-mais_2020
330,mamografia_70-ou-mais_2021
331,mamografia_70-ou-mais_2022


In [17]:
resultado

,REGIONAL DE SAÚDE,CLUSTER
0,alfenas,0
1,belo horizonte,1
2,coronel fabriciano,2
3,divinopolis,1
4,governador valadares,4
5,juiz de fora,1
6,leopoldina,3
7,montes claros,4
8,patos de minas,5
9,pouso alegre,1
